# Amazon ML Challenge 2026 — Stage 0: Colab & Central S3 Integrity Audit

In [1]:
# install dependencies
!pip uninstall -y -q datasets
!pip install -q -U \
    "s3fs" \
    "boto3" \
    "botocore" \
    "pyarrow" \
    "rapidfuzz" \
    "lightgbm" \
    "duckdb" \
    "scikit-learn" \
    "psutil" \
    "tqdm"

print("✅ All global competition dependencies installed cleanly with zero conflicts!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 156.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 10.0 MB/s eta 0:00:00
✅ All global competition dependencies installed cleanly with zero conflicts!


In [2]:
import os
import sys
import time
import importlib
import subprocess
import datetime

## 1. Environment & Dependency Audit

In [3]:
REQUIRED_PACKAGES = [
    "pandas",
    "numpy",
    "rapidfuzz",
    "lightgbm",
    "pyarrow",
    "boto3",
    "s3fs",
    "duckdb",
    "psutil",
]

print("🔍 [1/4] Checking required dependencies...")
missing_pkgs = []
for pkg in REQUIRED_PACKAGES:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing_pkgs.append(pkg)
if missing_pkgs:
    print(f"📦 Installing missing packages ({', '.join(missing_pkgs)})...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + missing_pkgs,
        check=True
    )
    print("✅ All dependencies installed successfully.")
else:
    print("✅ All required packages already present.")

# Import Verified Libraries
import psutil
import pandas as pd
import numpy as np
import s3fs

audit_results = {}

🔍 [1/4] Checking required dependencies...
✅ All required packages already present.


## 2. AWS Credentials & Authentication Audit

In [4]:
print("\n🔑 [2/4] Verifying AWS credentials from Colab Secrets...")
aws_access_key = None
aws_secret_key = None
aws_region = None
s3_bucket = None
# Attempt retrieval from Google Colab userdata secrets, with os.environ fallback
try:
    from google.colab import userdata
    aws_access_key = userdata.get("AWS_ACCESS_KEY_ID")
    aws_secret_key = userdata.get("AWS_SECRET_ACCESS_KEY")
    aws_region = userdata.get("AWS_DEFAULT_REGION") or "us-east-1"
    s3_bucket = userdata.get("S3_BUCKET_NAME")
except Exception:
    aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
    aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
    aws_region = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
    s3_bucket = os.environ.get("S3_BUCKET_NAME")
missing_secrets = []
if not aws_access_key: missing_secrets.append("AWS_ACCESS_KEY_ID")
if not aws_secret_key: missing_secrets.append("AWS_SECRET_ACCESS_KEY")
if not s3_bucket: missing_secrets.append("S3_BUCKET_NAME")
if missing_secrets:
    audit_results["AWS Credentials"] = "FAIL"
    raise RuntimeError(
        f"\n❌ Missing required Colab Secrets: {missing_secrets}\n"
        "-------------------------------------------------------------------------\n"
        "HOW TO FIX IN GOOGLE COLAB:\n"
        "1. Click the 'Secrets' (🔑 Key icon) in the left sidebar.\n"
        "2. Click '+ Add new secret' and enter the following keys:\n"
        "   - AWS_ACCESS_KEY_ID      (Your IAM access key)\n"
        "   - AWS_SECRET_ACCESS_KEY  (Your IAM secret key)\n"
        "   - AWS_DEFAULT_REGION     (e.g., us-east-1)\n"
        "   - S3_BUCKET_NAME         (e.g., your-competition-bucket)\n"
        "3. Crucial: Toggle the slider 'Notebook access' to ON for each secret.\n"
        "4. Re-run this cell.\n"
        "-------------------------------------------------------------------------"
    )
else:
    # Set environment variables for boto3 / AWS SDK compatibility
    os.environ["AWS_ACCESS_KEY_ID"] = aws_access_key
    os.environ["AWS_SECRET_ACCESS_KEY"] = aws_secret_key
    os.environ["AWS_DEFAULT_REGION"] = aws_region
    audit_results["AWS Credentials"] = "PASS"
    print(f"✅ Credentials loaded. Target S3 Bucket: {s3_bucket} (Region: {aws_region})")



🔑 [2/4] Verifying AWS credentials from Colab Secrets...
✅ Credentials loaded. Target S3 Bucket: amz-ml-crazy-dave-bucket-177683310295-us-east-1-an (Region: us-east-1)


## 3. S3 Read / Write Integrity & Latency Benchmark

In [5]:
print("\n⚡ [3/4] Performing S3 read/write ping test...")
bucket_name = s3_bucket.replace("s3://", "").strip("/")
ping_s3_path = f"s3://{bucket_name}/diagnostics/ping.parquet"
storage_opts = {
    "key": aws_access_key,
    "secret": aws_secret_key,
    "client_kwargs": {"region_name": aws_region}
}
try:
    fs = s3fs.S3FileSystem(**storage_opts)

    # Create sample diagnostic payload
    test_df = pd.DataFrame([
        {"test_id": "diag_001", "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(), "status": "HEALTHY"},
        {"test_id": "diag_002", "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(), "status": "HEALTHY"},
        {"test_id": "diag_003", "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(), "status": "HEALTHY"}
    ])

    # Benchmark Write
    t0 = time.perf_counter()
    test_df.to_parquet(ping_s3_path, engine="pyarrow", index=False, storage_options=storage_opts)
    write_ms = (time.perf_counter() - t0) * 1000

    # Benchmark Read
    t1 = time.perf_counter()
    read_df = pd.read_parquet(ping_s3_path, engine="pyarrow", storage_options=storage_opts)
    read_ms = (time.perf_counter() - t1) * 1000

    # Integrity check
    assert len(read_df) == len(test_df), f"Row mismatch: sent {len(test_df)}, received {len(read_df)}"
    assert list(read_df["test_id"]) == ["diag_001", "diag_002", "diag_003"], "Data content mismatch!"

    # Cleanup test artifact to maintain zero clutter
    if fs.exists(f"{bucket_name}/diagnostics/ping.parquet"):
        fs.rm(f"{bucket_name}/diagnostics/ping.parquet")

    audit_results["S3 Read/Write Integrity"] = f"PASS (W: {write_ms:.1f}ms | R: {read_ms:.1f}ms)"
    print(f"✅ S3 Round-trip verified. Write: {write_ms:.1f}ms | Read: {read_ms:.1f}ms (Diagnostic file cleaned)")
except Exception as e:
    audit_results["S3 Read/Write Integrity"] = "FAIL"
    raise RuntimeError(f"❌ S3 Read/Write audit failed: {str(e)}")



⚡ [3/4] Performing S3 read/write ping test...
✅ S3 Round-trip verified. Write: 479.7ms | Read: 723.4ms (Diagnostic file cleaned)


## 4. Hardware & Resource Audit

In [6]:
print("\n💻 [4/4] Auditing compute hardware & accelerators...")
# RAM & CPU
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
cpu_cores = os.cpu_count()
# GPU Detection
gpu_desc = "None (CPU Mode)"
try:
    smi_out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        stderr=subprocess.DEVNULL
    ).decode("utf-8").strip()
    if smi_out:
        gpu_desc = smi_out.replace("\n", "; ")
except Exception:
    pass
audit_results["Hardware Environment"] = f"RAM: {ram_gb:.1f}GB | {cpu_cores} vCPUs | GPU: {gpu_desc}"
# ------------------------------------------------------------------------------
# 5. Final PASS / FAIL Diagnostic Summary Table
# ------------------------------------------------------------------------------
print("\n" + "=" * 78)
print(f"{'AMAZON ML CHALLENGE 2026 — COLAB ENVIRONMENT AUDIT':^78}")
print("=" * 78)
print(f"  • Date & Time (UTC) : {datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  • Target S3 Bucket  : s3://{bucket_name}/")
print(f"  • Compute Hardware  : {cpu_cores} vCPUs | {ram_gb:.1f} GB RAM | {gpu_desc}")
print("-" * 78)
print(f"  {'AUDIT CHECK':<32} | {'STATUS / DETAILS':<41}")
print("-" * 78)
for check, status in audit_results.items():
    print(f"  {check:<32} | {status:<41}")
print("=" * 78)
print("🚀 Environment is 100% verified and ready for data streaming & training!\n")


💻 [4/4] Auditing compute hardware & accelerators...

              AMAZON ML CHALLENGE 2026 — COLAB ENVIRONMENT AUDIT              
  • Date & Time (UTC) : 2026-09-26 05:31:20
  • Target S3 Bucket  : s3://amz-ml-crazy-dave-bucket-177683310295-us-east-1-an/
  • Compute Hardware  : 44 vCPUs | 172.9 GB RAM | None (CPU Mode)
------------------------------------------------------------------------------
  AUDIT CHECK                      | STATUS / DETAILS                         
------------------------------------------------------------------------------
  AWS Credentials                  | PASS                                     
  S3 Read/Write Integrity          | PASS (W: 479.7ms | R: 723.4ms)           
  Hardware Environment             | RAM: 172.9GB | 44 vCPUs | GPU: None (CPU Mode)
🚀 Environment is 100% verified and ready for data streaming & training!

